[Guide Index](README.md)

# 02d Combining and Comparing Graphs

This guide covers combining graphs, finding how they differ, and the blank nodes that make that hard.

1. set operations
2. changes between graphs
3. matching blank nodes
4. `owl:imports`
5. skolemizing blank nodes


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run the setup cell below first, then the sections you want, from top to bottom within each section. Each section builds the data it needs; the setup cell defines the names they share.

In [1]:
from starlayer.graph import NamedNode, Namespace, StarLayerGraph, literal

EXL = Namespace("http://example.org/")

## 1. Set operations

A graph is a set of triples, so graphs can be combined and compared with operators. Each one returns a new graph, and the graphs you started with are not changed:

- `a + b` (union) has the triples that are in either graph.
- `a - b` (difference) has the triples in `a` that are not in `b`.
- `a & b` (intersection) has the triples that are in both.
- `a ^ b` (symmetric difference) has the triples that are in one graph but not both.

The in-place forms `a += b`, `a -= b`, `a &= b` and `a ^= b` modify `a` itself, and leave `b` as it is.

Blank nodes are compared by their labels, so a blank node in one graph is not the same as a blank node in the other unless they have the same label. How to compare graphs that have blank nodes, is in section 3.

Both sides must be the same kind of object: two graphs, or two datasets. A dataset is a set of quads, so its operators compare the graph name as well as the triple: the same triple in two different graphs is two different quads. To combine a graph with a graph inside a dataset, obtain that graph with `dataset.graph(name)`. It remains part of the dataset, so changing it with `+=` changes the dataset.

In [9]:
from starlayer.graph import StarLayerDataset

g = StarLayerGraph(prefixes={"ex": "http://example.org/"})
g.parse(data="@prefix ex: <http://example.org/> . ex:a ex:knows ex:b ; ex:age 4 . ex:b ex:knows ex:c .", format="turtle")

other = StarLayerGraph(prefixes={"ex": "http://example.org/"})
other.add((EXL.a, EXL.knows, EXL.b))
other.add((EXL.c, EXL.knows, EXL.d))

print("g:")
print(g.serialize(format="turtle", only_used=True))
print("other:")
print(other.serialize(format="turtle", only_used=True))

print("g - other: the triples in g that are not in other")
print((g - other).serialize(format="turtle", only_used=True))
print("g & other: the triples in both")
print((g & other).serialize(format="turtle", only_used=True))
print("g + other: the triples in either")
print((g + other).serialize(format="turtle", only_used=True))
print("g ^ other: the triples in one but not both")
print((g ^ other).serialize(format="turtle", only_used=True))

print("A graph inside a dataset can be combined with a graph")
dataset = StarLayerDataset(prefixes={"ex": "http://example.org/"})
inside = dataset.graph(EXL.people)  # the graph named ex:people, part of the dataset
inside.add((EXL.c, EXL.knows, EXL.e))
print((g + inside).serialize(format="turtle", only_used=True))

print("+= changes the graph inside the dataset, so the dataset changes")
inside += other
print(dataset.serialize(format="trig", only_used=True))

g:
@prefix ex: <http://example.org/> .
ex:b ex:knows ex:c .
ex:a ex:age 4 ;
	ex:knows ex:b .

other:
@prefix ex: <http://example.org/> .
ex:c ex:knows ex:d .
ex:a ex:knows ex:b .

g - other: the triples in g that are not in other
@prefix ex: <http://example.org/> .
ex:b ex:knows ex:c .
ex:a ex:age 4 .

g & other: the triples in both
@prefix ex: <http://example.org/> .
ex:a ex:knows ex:b .

g + other: the triples in either
@prefix ex: <http://example.org/> .
ex:c ex:knows ex:d .
ex:b ex:knows ex:c .
ex:a ex:age 4 ;
	ex:knows ex:b .

g ^ other: the triples in one but not both
@prefix ex: <http://example.org/> .
ex:c ex:knows ex:d .
ex:b ex:knows ex:c .
ex:a ex:age 4 .

A graph inside a dataset can be combined with a graph
@prefix ex: <http://example.org/> .
ex:c ex:knows ex:e .
ex:b ex:knows ex:c .
ex:a ex:age 4 ;
	ex:knows ex:b .

+= changes the graph inside the dataset, so the dataset changes
@prefix ex: <http://example.org/> .
ex:people {
	ex:a ex:knows ex:b .
	ex:c ex:knows ex:d , ex

## 2. Changes between graphs

`diff` is a `StarLayer` function. It compares two graphs and returns an object, a `GraphDelta`, that holds the triples to add and the triples to remove to turn `old` into `new`. The delta has its own methods:

| Name | What it is | What it does |
| --- | --- | --- |
| `diff(old, new)` | a function in `starlayer.graph` | compares two graphs and returns a `GraphDelta` |
| `delta.added`, `delta.removed` | attributes of a delta | the triples to add and the triples to remove |
| `len(delta)`, `bool(delta)` | Python functions applied to a delta | `len` is the number of triples in the delta (added and removed). An empty delta is false, so `if diff(a, b):` asks whether the graphs differ |
| `delta.apply(graph)` | a method of a delta | changes the graph in place and returns it |
| `delta.applied_to(graph)` | a method of a delta | returns a new graph: a copy of `graph` with the delta applied. `graph` itself is not changed |
| `delta.inverse()` | a method of a delta | returns a new delta that undoes this one. Apply it to a graph to undo the change |
| `deltaA.then(deltaB)` | a method of a delta (`deltaA` and `deltaB` are deltas) | returns a new delta for `deltaA` followed by `deltaB`: a triple that `deltaA` adds and `deltaB` removes is no change |


Blank nodes are compared by their labels, as in set operations. Two graphs that differ only in the labels of their blank nodes give a delta that removes and adds every triple that has a blank node. How to compare graphs that have blank nodes, is in section 3.

A delta compares triples and does not retain graph names so `diff` and `apply` can only be used with graphs and reject use of a dataset with a `TypeError`. 

In [10]:
from starlayer.graph import diff

old = StarLayerGraph()
new = StarLayerGraph()
old.parse(data="@prefix ex: <http://example.org/> . ex:a ex:p ex:b ; ex:q ex:b .", format="turtle")
new.parse(data="@prefix ex: <http://example.org/> . ex:a ex:p ex:c ; ex:q ex:b .", format="turtle")
original = old.copy()  # kept to check that the change can be undone

print("1. diff: the triples to add and the triples to remove")
delta = diff(old, new)  # diff is a function; it returns a GraphDelta, here called delta
for triple in delta.added:
    print("   add:   ", *(old.qname_term(term) for term in triple))
for triple in delta.removed:
    print("   remove:", *(old.qname_term(term) for term in triple))
print("   the delta is true when the graphs differ:", bool(delta), "| for two equal graphs:", bool(diff(old, original)))

print("\n2. applied_to: a copy of old with the change made, without changing old")
preview = delta.applied_to(old)
print(preview.serialize(format="turtle", only_used=True))
print("   the copy equals new:", set(preview.triples()) == set(new.triples()), "| old is unchanged:", set(old.triples()) == set(original.triples()))

print("\n3. apply: changes old itself")
delta.apply(old)  # apply is a method of delta; old is the graph it changes
print("   old now equals new:", set(old.triples()) == set(new.triples()))

print("\n4. inverse: the delta that undoes it")
delta.inverse().apply(old)  # inverse is a method of delta: it returns a second delta, and that one is applied
print("   old equals the original again:", set(old.triples()) == set(original.triples()))

print("\n5. then: one delta followed by another")
newer = new.copy()
newer.add((EXL.a, EXL.r, EXL.d))
print("   old to new, then new to newer, is the same as old to newer:", delta.then(diff(new, newer)) == diff(old, newer))

1. diff: the triples to add and the triples to remove
   add:    ex:a ex:p ex:c
   remove: ex:a ex:p ex:b
   the delta is true when the graphs differ: True | for two equal graphs: False

2. applied_to: a copy of old with the change made, without changing old
@prefix ex: <http://example.org/> .
ex:a ex:p ex:c ;
	ex:q ex:b .

   the copy equals new: True | old is unchanged: True

3. apply: changes old itself
   old now equals new: True

4. inverse: the delta that undoes it
   old equals the original again: True

5. then: one delta followed by another
   old to new, then new to newer, is the same as old to newer: True


## 3. Matching blank nodes

A blank node has no name, only a label that means nothing outside its own graph.  StarLayer gives a document's blank nodes new labels eacn time it is parsed. The `diff` function (section 2) and the set operators (section 1) compare label of nodes.  As a result, the same or similar graphs containing blank nodes may look completely different. 

`isomorphic` ([Canonical hashing](07c-canonical-hashing.ipynb)) is a able to check that two graphs are the same apart from blank node labels, but not how they differ. 

`align_blank_nodes(graph, to=reference)` is a function that aligns the blank node labels of two graphs so that `diff` and set fucntions identify real differences. Two blank nodes are paired when they have the same content and context (the same parent), or when an edited blank node retains a specified number of its edges.   `alignment.graph` is the graph to give to `diff`, and `alignment.pairs` shows how each blank node was paired.

In [ ]:
from starlayer.graph import align_blank_nodes, diff, isomorphic

# two similar graphs containing blank nodes
prefix = "@prefix ex: <http://example.org/> . "
old = StarLayerGraph()
old.parse(data=prefix + 'ex:alice ex:hasPet [ a ex:Cat ; ex:name "Tabby" ] . ex:bob ex:hasPet [ a ex:Cat ; ex:name "Tabby" ] .', format="turtle")
new = StarLayerGraph()
new.parse(data=prefix + 'ex:alice ex:hasPet [ a ex:Cat ; ex:name "Tabby" ] . ex:bob ex:hasPet [ a ex:Cat ; ex:name "Felix" ] .', format="turtle")

# calculated differences due to blank nodes
print("1. diff with the blank labels unaligned: every triple of a cat differs")
plain = diff(old, new)
removed, added = StarLayerGraph(prefixes=old.prefixes), StarLayerGraph(prefixes=old.prefixes)
removed.extend(plain.removed)
added.extend(plain.added)
print("   changes:", len(plain), "(the blank node labels in the two lists are not related)")
print("   removed:\n" + removed.serialize(format="turtle", only_used=True, bnode_labels="short").split("\n", 1)[1])
print("   added:\n" + added.serialize(format="turtle", only_used=True, bnode_labels="short").split("\n", 1)[1])

# create alignment graph and recalculate diff
alignment = align_blank_nodes(new, to=old)
delta = diff(old, alignment.graph)
print("\n2. after align_blank_nodes, diff finds the one change")
removed, added = StarLayerGraph(prefixes=old.prefixes), StarLayerGraph(prefixes=old.prefixes)
removed.extend(delta.removed)
added.extend(delta.added)
print("   removed:", removed.serialize(format="turtle", only_used=True, bnode_labels="short").split("\n", 1)[1].strip())
print("   added:  ", added.serialize(format="turtle", only_used=True, bnode_labels="short").split("\n", 1)[1].strip())
print("   applying the delta to old gives a graph isomorphic to new:", isomorphic(delta.apply(old.copy()), new))



1. diff with the blank labels unaligned: every triple of a cat differs
   changes: 12 (the blank node labels in the two lists are not related)
   removed:
ex:alice ex:hasPet _:b0 .
ex:bob ex:hasPet _:b1 .
_:b0 ex:name "Tabby" ;
	a ex:Cat .
_:b1 ex:name "Tabby" ;
	a ex:Cat .

   added:
ex:alice ex:hasPet _:b1 .
ex:bob ex:hasPet _:b0 .
_:b0 ex:name "Felix" ;
	a ex:Cat .
_:b1 ex:name "Tabby" ;
	a ex:Cat .


2. after align_blank_nodes, diff finds the one change
   removed: _:b0 ex:name "Tabby" .
   added:   _:b0 ex:name "Felix" .
   applying the delta to old gives a graph isomorphic to new: True


## 4. `owl:imports`

`owl:imports` is a way to say that one document depends on another: `<A> owl:imports <B>` means that to use graph A you also need the triples of graph B. `owl:imports` comes from OWL, where an ontology imports other ontologies, but it can be used for any RDF document that is built from parts: a shapes graph that imports shared shapes, or a SKOS vocabulary that imports another SKOS vocabulary. StarLayer can follow any `owl:imports` triple, whatever kind of document imports or is imported.

The IRI after `owl:imports` names the document to be imported. It is often the URL of the location of the document to be fetched, but it need not be. StarLayer never fetches an import on its own. The application supplies a `loader`, a function that takes the IRI of one import and returns that document as a graph (or `None`).

- `graph.owl_imports()` lists the IRIs a graph imports directly (does not include imports of imports). It fetches nothing.
- `graph.resolve_imports(loader=...)` follows the imports transitively (includes imports of imports) and returns a new graph with the new triples merged. `mode="delta"` gives a graph of only what was added, and `mode="in-place"` adds the new triples to the original graph. An import that is already present, or that the loader cannot retrieve, is skipped.

An imported document normally describes itself under the IRI it is imported by, with a header such as `<http://example.org/base> a owl:Ontology`. StarLayer uses that to see that an import is already present. A document with no such header is fetched each time `resolve_imports` is called.

In [15]:
BASE = NamedNode("http://example.org/base")
onto = StarLayerGraph()
onto.parse(data="<http://example.org/onto> <http://www.w3.org/2002/07/owl#imports> <http://example.org/base> .", format="turtle")

# the loader is given the IRI of an import and returns that document as a graph, or None
def loader(iri):  # a real loader would read a file or fetch a URL
    if iri == BASE:
        found = StarLayerGraph()
        found.parse(data="<http://example.org/base> a <http://www.w3.org/2002/07/owl#Ontology> .", format="turtle")
        return found
    return None

print("1. the imports of onto:", [str(i) for i in onto.owl_imports()])

print("\n2. resolve_imports: onto with the imported triples merged in")
merged = onto.resolve_imports(loader=loader)
print("   triples:", len(onto), "before,", len(merged), "after")

print("\n3. mode='delta': only the triples that were added")
added = onto.resolve_imports(loader=loader, mode="delta")
print("   triples:", len(added))

1. the imports of onto: ['<http://example.org/base>']

2. resolve_imports: onto with the imported triples merged in
   triples: 1 before, 2 after

3. mode='delta': only the triples that were added
   triples: 1


## 5. Skolemizing blank nodes

`graph.skolemize()` returns a new graph in which each blank node used as a subject or object (inside triple terms too) is replaced by a skolem IRI, a new IRI that names it, so that it can be referred to from outside. `authority=` and `basepath=` choose the IRI, and `bnode=` replaces only that blank node. 

`graph.de_skolemize()` turns skolem IRIs back into blank nodes.

A skolem IRI is made from the blank node's label, so it names the node within this graph. To compare two graphs that have blank nodes, see section 3.

In [16]:
from starlayer.graph import BlankNode

bn = StarLayerGraph(prefixes={"ex": "http://example.org/"})
bn.parse(data="@prefix ex: <http://example.org/> . ex:a ex:p [ ex:q 1 ] .", format="turtle")

named = bn.skolemize(authority="http://example.org", basepath="/.well-known/genid/")
print("skolem IRIs:", all(o.value.startswith("http://example.org/.well-known/genid/") for _, _, o in named.triples((None, EXL.p, None))))
restored = named.de_skolemize()
print("blank nodes again:", all(isinstance(o, BlankNode) for _, _, o in restored.triples((None, EXL.p, None))))

skolem IRIs: True
blank nodes again: True
